# 电桥法测量电阻计算器 (Wheatstone Bridge)

本 Notebook 用于通过**惠斯通电桥（单臂电桥）**测量未知电阻值 $R_x$。包含不同比例臂（$1:1, 1:10, 10:1$ 等）下的平衡电阻统计、电桥灵敏度 $S$ 测定、仪器误差限 $\Delta_{\mathrm{inst}}$ 评估、不确定度合成及各比例臂对比汇总。

---
### 实验原理与数学公式

#### 1. 惠斯通电桥平衡条件
当检流计指针示数为零时，电桥达到平衡：
$$R_x = R_0 \cdot \frac{R_1}{R_2} = R_0 \cdot C$$
* $R_x$：待测电阻
* $R_0$：比较臂标准电阻箱读数
* $C = R_1 / R_2$：比例臂倍率（如 $1, 0.1, 10$）

#### 2. 电桥灵敏度 $S$ 与仪器误差限 $\Delta_{\mathrm{inst}}$
平衡后微调比较臂电阻使检流计偏转 $\Delta n$ 格（通常取 5 格），记偏调后的读数为 $R_0'$：
$$\Delta R_0 = |R_0' - R_0|, \quad S = \frac{\Delta n}{\Delta R_0 / R_0} = \frac{\Delta n \cdot R_0}{\Delta R_0}$$
电桥所能分辨的最小电阻改变量（即由灵敏度决定的仪器误差限）为：
$$\Delta_{\mathrm{inst}} = \frac{R_0}{S} = \frac{\Delta R_0}{\Delta n}$$

#### 3. 不确定度传递与合成公式
比例臂 $C$ 由精密标准电阻构成，其不确定度可忽略，则：
* $R_0$ 的合成不确定度：
$$u(R_0) = \sqrt{ u_A^2(R_0) + u_B^2(R_0) }, \quad u_A(R_0) = \frac{s(R_0)}{\sqrt{n}}, \quad u_B(R_0) = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$
* 待测电阻 $R_x$ 的不确定度：
$$u(R_x) = C \cdot u(R_0), \quad u_r(R_x) = \frac{u(R_x)}{R_x} = \frac{u(R_0)}{R_0}$$
* 结果按照“四舍六入五凑偶”保留一位有效数字，末位对齐。

In [ ]:

from decimal import Decimal
from python.utils import scientific_round, calculate_stats

print("电桥法计算模块加载完成。")

### 1. 测量实验数据配置 (支持多组标称值与比例臂)
> **提示**：可以在 `calibration_data` 列表中增减测试组。每组包含：标称值、比例臂标签、比例因子 $C$、5次平衡读数 $R_0$ 列表，以及偏转 5 格时的读数 $R_0'$。

In [ ]:
# 实验测量数据配置列表
# 结构: dict(R_nominal, ratio_label, C, R0_list, R0_prime)
calibration_data = [
    {
        "R_nominal": "10000", "ratio_label": "1:1", "C": Decimal("1"),
        "R0_list": [Decimal("9985.2"), Decimal("9985.0"), Decimal("9985.3"), Decimal("9985.1"), Decimal("9985.2")],
        "R0_prime": Decimal("9995.2")
    },
    {
        "R_nominal": "10000", "ratio_label": "10:1", "C": Decimal("10"),
        "R0_list": [Decimal("998.5"), Decimal("998.4"), Decimal("998.6"), Decimal("998.5"), Decimal("998.5")],
        "R0_prime": Decimal("1002.5")
    },
    {
        "R_nominal": "2000", "ratio_label": "1:1", "C": Decimal("1"),
        "R0_list": [Decimal("1996.8"), Decimal("1996.9"), Decimal("1996.7"), Decimal("1996.8"), Decimal("1996.8")],
        "R0_prime": Decimal("2002.8")
    },
    {
        "R_nominal": "2000", "ratio_label": "10:1", "C": Decimal("10"),
        "R0_list": [Decimal("199.7"), Decimal("199.6"), Decimal("199.7"), Decimal("199.8"), Decimal("199.7")],
        "R0_prime": Decimal("201.2")
    },
    {
        "R_nominal": "200", "ratio_label": "1:1", "C": Decimal("1"),
        "R0_list": [Decimal("199.65"), Decimal("199.62"), Decimal("199.68"), Decimal("199.64"), Decimal("199.65")],
        "R0_prime": Decimal("201.15")
    },
    {
        "R_nominal": "200", "ratio_label": "1:10", "C": Decimal("0.1"),
        "R0_list": [Decimal("1996.4"), Decimal("1996.2"), Decimal("1996.5"), Decimal("1996.4"), Decimal("1996.3")],
        "R0_prime": Decimal("2005.4")
    },
    {
        "R_nominal": "200", "ratio_label": "10:1", "C": Decimal("10"),
        "R0_list": [Decimal("19.96"), Decimal("19.95"), Decimal("19.97"), Decimal("19.96"), Decimal("19.96")],
        "R0_prime": Decimal("20.36")
    }
]

print(f"成功载入 {len(calibration_data)} 组电桥测量方案。")

### 2. 批量计算、灵敏度分析与不确定度评定

In [ ]:
delta_n = Decimal("5")  # 偏转格数
results = []

for item in calibration_data:
    R_nom = item["R_nominal"]
    ratio = item["ratio_label"]
    C = item["C"]
    R0_data = item["R0_list"]
    R0_prime = item["R0_prime"]
    
    R0_mean_temp = sum(R0_data) / Decimal(len(R0_data))
    delta_R0 = abs(R0_prime - R0_mean_temp)
    
    if delta_R0 == 0:
        S = Decimal("Infinity")
        delta_inst = Decimal("0")
    else:
        S = delta_n * R0_mean_temp / delta_R0
        delta_inst = R0_mean_temp / S
        
    R0_mean, u_R0, u_a_R0, u_b_R0, _ = calculate_stats(R0_data, str(delta_inst))
    
    Rx_mean = C * R0_mean
    u_Rx = C * u_R0
    u_r_Rx = u_Rx / Rx_mean if Rx_mean != 0 else Decimal("0")
    Rx_final, u_Rx_final = scientific_round(Rx_mean, u_Rx)
    
    results.append({
        "R_nominal": R_nom, "ratio_label": ratio, "C": C,
        "R0_mean": R0_mean, "S": S, "delta_inst": delta_inst,
        "Rx_mean": Rx_mean, "u_Rx": u_Rx, "u_r_Rx": u_r_Rx,
        "Rx_final": Rx_final, "u_Rx_final": u_Rx_final
    })

print("=" * 85)
print("                             测 量 结 果 汇 总 表                              ")
print("=" * 85)
header = f"{'标定电阻':>8} {'比例臂':>8} {'R0均值(Ω)':>12} {'灵敏度S':>10} {'Δ_inst(Ω)':>12} {'Rx(Ω)':>12} {'u(Rx)(Ω)':>10} {'相对不确定度':>10}"
print(header)
print("-" * 85)
for res in results:
    S_str = f"{res['S']:.1f}" if res['S'] != Decimal("Infinity") else "∞"
    print(f"{res['R_nominal'] + ' Ω':>8} {res['ratio_label']:>8} {res['R0_mean']:>12.4f} {S_str:>10} "
          f"{res['delta_inst']:>12.4f} {res['Rx_mean']:>12.4f} {res['u_Rx']:>10.4f} {res['u_r_Rx']*100:>9.2f}%")
print("=" * 85)

print("\n" + "*" * 55)
print("             最 终 规 范 修 约 结 果            ")
print("*" * 55)
for res in results:
    print(f"标定 {res['R_nominal']:>5} Ω (比例臂 {res['ratio_label']:>4}) : Rx = {res['Rx_final']} ± {res['u_Rx_final']} Ω")
print("*" * 55)